# 第 35 章 · 量化与导出：模型变小、脱离 Python 部署

本章回答：量化是怎么把 float 变成 int8 / int4 的，误差从哪来？weight-only 和动态激活量化有什么不同？`torch.export` 导出的是什么，AOTInductor 又把它变成了什么？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/35-quant-export.md`（讲解、动画、练习都在讲义里）。对应原文：AISPE ch13「PyTorch 架构优化（torchao）、量化、稀疏性和剪枝」（`ch13.md` 第 997–1006 行）；torchao 文档与 README（〔在线〕github.com/pytorch/ao）；官方文档 `torch.export`、AOTInductor（〔在线〕）；LLM 全链路地图量化部分（`llm-full-stack/notes/00-full-stack-map.md`）；本章量化误差、文件大小和导出结果是本机 torch 2.15.0 + torchao 0.18.0 实测

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import io
import os
import tempfile

import torch
import torch.nn as nn

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 对称量化 | s = max|w|/127 | `symmetric_quant([-0.6, 0.2, 1.0], 8)` | [−76, 25, 127]，还原 [−0.598, 0.197, 1.0] |
| 第一步 int4 | s = max|w|/7 | `symmetric_quant(..., 4)` | [−4, 1, 7] |
| 第二步 torchao | Int8WeightOnlyConfig | `int8_weight_only()` | 8.01 → 2.02 MB，误差 0.56%，Int8Tensor |
| 第三、四步 export + AOTInductor | — | `export_and_aoti()` | linear / gelu / linear；batch 5；一致 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `export_and_aoti`**（来自 `ch35_quant_export.py`）：下面要用到的小工具。

In [ ]:
def export_and_aoti():
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(16, 32), nn.GELU(), nn.Linear(32, 4)).eval()
    x = torch.randn(2, 16)
    batch = torch.export.Dim("batch", min=1, max=64)
    ep = torch.export.export(m, (x,), dynamic_shapes={"input": {0: batch}})
    ops = [n.target.__name__ for n in ep.graph.nodes if n.op == "call_function"]
    path = torch._inductor.aoti_compile_and_package(ep, package_path=os.path.join(tempfile.gettempdir(), "ptdd_ch35.pt2"))
    runner = torch._inductor.aoti_load_package(path)
    return ops, tuple(runner(torch.randn(5, 16)).shape), torch.allclose(runner(x), m(x), atol=1e-5)

**函数 `state_mb`**（来自 `ch35_quant_export.py`）：下面要用到的小工具。

In [ ]:
def state_mb(m: nn.Module) -> float:
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    return buf.tell() / 2**20

**函数 `int8_weight_only`**（来自 `ch35_quant_export.py`）：下面要用到的小工具。

In [ ]:
def int8_weight_only():
    from torchao.quantization import Int8WeightOnlyConfig, quantize_

    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(1024, 1024), nn.ReLU(), nn.Linear(1024, 1024)).eval()
    x = torch.randn(4, 1024)
    ref = m(x)
    before = state_mb(m)
    quantize_(m, Int8WeightOnlyConfig())
    y = m(x)
    return round(before, 2), round(state_mb(m), 2), ((y - ref).norm() / ref.norm()).item(), type(m[0].weight).__name__

**函数 `symmetric_quant`**（来自 `ch35_quant_export.py`）：对称量化：s = max|w| / (2^(b−1) − 1)，q = round(w / s)，还原 q × s。

In [ ]:
def symmetric_quant(w, bits: int):
    """对称量化：s = max|w| / (2^(b−1) − 1)，q = round(w / s)，还原 q × s。"""
    w = torch.tensor(w)
    s = w.abs().max() / (2 ** (bits - 1) - 1)
    q = torch.round(w / s)
    return s.item(), q.int().tolist(), (q * s).tolist()

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 [-0.6, 0.2, 1.0]：int8", symmetric_quant([-0.6, 0.2, 1.0], 8), "\n                    int4", symmetric_quant([-0.6, 0.2, 1.0], 4))

In [ ]:
print("\n第二步 int8 weight-only（前 MB, 后 MB, 相对误差, 权重类型）：", int8_weight_only())

In [ ]:
print("\n第三、四步 export 的算子 / AOTInductor 跑 batch 5 / 与原模型一致：", export_and_aoti())

## 3. 代码里才看得到的

1. **正好落在两个整数中间的值不稳**：−0.5 ÷ (1/7) = −3.5，float32 下得 −3、float64 下得 −4，所以例子换成了 −0.6。
2. **torchao 的配置名随版本变**：本机是 0.18.0。

## 4. 自己改一改

- `symmetric_quant([0.01, 0.02, 5.0], 8)`，看小数被量化成什么。
- 把 `Int8WeightOnlyConfig` 换成 `Int8DynamicActivationInt8WeightConfig`，看误差。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 量化是粗尺子加系数：s 等于最大绝对值除以 127，存 round(w/s)，用时乘回 s；
> 位数越少格子越粗，int4 要分组；decode 吃带宽，weight-only 最划算；
> export 抓整图、抓不下就报错，Dim 声明会变的维；AOTInductor 提前编成 .pt2，C++ 也能跑。